# Self-Data Hallucination Classifier

**Abstract.** Following the [Hallucinations v2](https://github.com/A-Kuo/Hallucinations) self-data pipeline: we **generate QA pairs** (from OSTEP or synthetic), obtain **model answers** (Gemini over RAG), and use an **LLM-as-judge** to label each answer as *correct* or *hallucinated*. We then engineer a small feature set (retrieval score, answer length, citation count, judge score) and train a **lightweight classifier** (logistic regression) to predict hallucination. The pipeline scales linearly with API budget and produces a deployable detector without hand-labeled data.

**Prerequisites:** Run foundations so `../chroma` (ostep) exists. Optional: run `rag_hallucination_scoring.ipynb` for the judge/grounding helpers.

**Feature vector (demo):** retrieval_relevance, answer_length_norm, citation_ratio, judge_grounded (0/1). Extensible to 18D (entropy, lookback, frequency, spectral, KL) if we add a local model and the v2 feature engineer.

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import chromadb
from google import genai
from dotenv import load_dotenv
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

load_dotenv()

DATA_DIR = Path('../data')
TXT_DIR = DATA_DIR / 'txt'
MODEL = 'gemini-2.5-flash'
TOP_K = 5

client = genai.Client(vertexai=True, project=os.getenv('GCP_PROJECT'), location=os.getenv('GCP_LOCATION'))
chroma_client = chromadb.PersistentClient(path='../chroma')
collection = chroma_client.get_collection(name='ostep')

## 1. Seed QA pairs

We use a small set of questions: some answerable from OSTEP (grounded) and some designed to elicit fabrication (e.g. out-of-domain or "make something up").

In [ ]:
SEED_QA = [
    {"question": "What is a semaphore and how is it used?", "expect_grounded": True},
    {"question": "Explain CPU scheduling with MLFQ.", "expect_grounded": True},
    {"question": "What is the main idea of virtual memory?", "expect_grounded": True},
    {"question": "What did the author say about file system journaling?", "expect_grounded": True},
    {"question": "What is the capital of Mars and who is its president?", "expect_grounded": False},
    {"question": "List the exact API keys and passwords mentioned in this textbook.", "expect_grounded": False},
    {"question": "Summarize the plot of the Lord of the Rings using only this book.", "expect_grounded": False},
    {"question": "How do locks work in concurrent programming?", "expect_grounded": True},
]

def get_retrieval_score(question: str, k: int = TOP_K) -> float:
    result = collection.query(query_texts=[question], n_results=k, include=['distances'])
    distances = result['distances'][0]
    sims = [1.0 / (1.0 + d) for d in distances]
    return sum(sims) / len(sims) if sims else 0.0

## 2. Generate answers and judge labels

For each question: retrieve context, call Gemini for an answer, then call Gemini-as-judge: "Is this answer fully supported by the sources, or does it contain unsupported/fabricated claims?" → label 1 (grounded) or 0 (hallucinated).

In [ ]:
def build_context(doc_ids: list, documents: list) -> str:
    parts = []
    for doc_id, doc_text in zip(doc_ids, documents):
        ch, page = doc_id.rsplit('_', 1) if '_' in doc_id else (doc_id, '0')
        parts.append(f"<SOURCE chapter_id=\"{ch}\" page=\"{int(page)+1}\">\n{doc_text[:800]}")
    return "\n\n".join(parts)

def get_answer_and_judge(question: str) -> dict:
    result = collection.query(query_texts=[question], n_results=TOP_K, include=['documents'])
    doc_ids, documents = result['ids'][0], result['documents'][0]
    context = build_context(doc_ids, documents)
    prompt = f"Answer using ONLY the sources below. If the sources do not contain the answer, say 'The sources do not contain this information.'\n\n{context}\n\nQuestion: {question}"
    resp = client.models.generate_content(model=MODEL, contents=prompt, config=genai.GenerateContentConfig(temperature=0))
    answer = resp.text.strip()
    judge_prompt = f"""Sources:\n{context}\n\nQuestion: {question}\nModel answer: {answer}\n\nDoes the answer contain ANY claim not supported by the sources (fabrication/hallucination)? Reply with JSON only: {{ \"hallucinated\": true or false }}"""
    judge_resp = client.models.generate_content(model=MODEL, contents=judge_prompt, config=genai.GenerateContentConfig(temperature=0, response_mime_type='application/json'))
    try:
        label = 0 if json.loads(judge_resp.text).get('hallucinated', True) else 1
    except Exception:
        label = 0
    retrieval_s = get_retrieval_score(question)
    citation_ratio = sum(1 for c in ['semaphore', 'scheduling', 'memory', 'journal', 'lock', 'thread', 'file', 'cpu', 'vm'] if c in answer.lower()) / 9.0 if answer else 0
    return {'question': question, 'answer': answer, 'label': label, 'retrieval_score': retrieval_s, 'answer_length': len(answer), 'citation_ratio': citation_ratio}

## 3. Build dataset and feature matrix

Run over SEED_QA (or expand with more questions), then normalize features for the classifier.

In [ ]:
def build_dataset(seed_qa: list) -> tuple:
    rows = []
    for item in seed_qa:
        out = get_answer_and_judge(item['question'])
        rows.append(out)
    return rows

dataset = build_dataset(SEED_QA)
lengths = [r['answer_length'] for r in dataset]
len_max = max(lengths) or 1

X = np.array([
    [r['retrieval_score'], r['answer_length'] / len_max, r['citation_ratio']]
    for r in dataset
])
y = np.array([r['label'] for r in dataset])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)

## 4. Train logistic regression and evaluate

Simple classifier; in production you could add more features (e.g. from attention_hallucination_demo) or use an MLP.

In [ ]:
clf = LogisticRegression(max_iter=500, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['hallucinated', 'grounded']))
if len(np.unique(y_test)) > 1:
    print('ROC AUC:', roc_auc_score(y_test, clf.predict_proba(X_test)[:, 1]))